# File Formats

I present three data formats, feather, parquet and hdf but it exists several more like [Apache Avro](http://avro.apache.org/docs/current/) or [Apache ORC](https://orc.apache.org).

These data formats may be more appropriate in certain situations.
However, the software needed to handle them is either more difficult
to install, incomplete, or more difficult to use because less
documentation is provided. For ORC and AVRO the python libraries
offered are less well maintained than the formats we will see. You can find many on
the web but it is hard to know which one is the most stable.
- [pyorc](https://github.com/noirello/pyorc)
- [avro](https://avro.apache.org/docs/1.10.0/gettingstartedpython.html) and [fastavro](https://github.com/fastavro/fastavro)
The following formats are supported
by pandas and apache arrow. These softwares are supported by very strong communities.

## Feather

For light data, it is recommanded to use [Feather](https://github.com/wesm/feather). It is a fast, interoperable data frame storage that comes with bindings for python and R.

Feather uses also the Apache Arrow columnar memory specification to represent binary data on disk. This makes read and write operations very fast.

## Parquet file format

[Parquet format](https://github.com/apache/parquet-format) is a common binary data store, used particularly in the Hadoop/big-data sphere. It provides several advantages relevant to big-data processing:

The Apache Parquet project provides a standardized open-source columnar storage format for use in data analysis systems. It was created originally for use in Apache Hadoop with systems like Apache Drill, Apache Hive, Apache Impala, and Apache Spark adopting it as a shared standard for high performance data IO.



## Hierarchical Data Format

 [HDF](https://en.wikipedia.org/wiki/Hierarchical_Data_Format) is a self-describing data format
allowing an application to interpret the structure and
contents of a file with no outside information.
One HDF file can hold a mix of related objects
which can be accessed as a group or as individual objects.

Let's create some big dataframe with consitent data (Floats) and 10% of missing values:

In [1]:
!pip install pyarrow pandas lorem

from pyarrow import feather
import pandas as pd
import numpy as np
arr = np.random.randn(500000) # 10% nulls
arr[::10] = np.nan
df = pd.DataFrame({'column_{0}'.format(i): arr for i in range(10)})

In [2]:
%time df.to_csv('test.csv')

CPU times: user 10.2 s, sys: 142 ms, total: 10.3 s
Wall time: 13 s


In [3]:
!rm -f test.h5

In [4]:
%time df.to_hdf("test.h5", key="test")

CPU times: user 264 ms, sys: 60.8 ms, total: 324 ms
Wall time: 527 ms


In [5]:
%time df.to_parquet('test.parquet')

CPU times: user 420 ms, sys: 54.4 ms, total: 474 ms
Wall time: 861 ms


In [6]:
%time df.to_feather('test.feather')

CPU times: user 168 ms, sys: 45 ms, total: 213 ms
Wall time: 254 ms


In [7]:
!du -sh test.*

88M	test.csv
36M	test.feather
42M	test.h5
38M	test.parquet


In [8]:
%%time
df = pd.read_csv("test.csv")
len(df)

CPU times: user 1.43 s, sys: 107 ms, total: 1.54 s
Wall time: 2.41 s


500000

In [9]:
%%time
df = pd.read_hdf("test.h5")
len(df)

CPU times: user 44.9 ms, sys: 25.9 ms, total: 70.8 ms
Wall time: 101 ms


500000

In [10]:
%%time
df = pd.read_parquet("test.parquet")
len(df)

CPU times: user 85.1 ms, sys: 80.8 ms, total: 166 ms
Wall time: 251 ms


500000

In [11]:
%%time
df = pd.read_feather("test.feather")
len(df)

CPU times: user 28.1 ms, sys: 42.8 ms, total: 70.9 ms
Wall time: 83.1 ms


500000

In [12]:
# Now we create a new big dataframe with a column of strings

In [13]:
import numpy as np
import pandas as pd
from lorem import sentence

words = np.array(sentence().strip().lower().replace(".", " ").split())

# Set the seed so that the numbers can be reproduced.
np.random.seed(0)
n = 1000000
df = pd.DataFrame(np.c_[np.random.randn(n, 5),
                  np.random.randint(0,10,(n, 2)),
                  np.random.randint(0,1,(n, 2)),
np.array([np.random.choice(words) for i in range(n)])] ,
columns=list('ABCDEFGHIJ'))

df["A"][::10] = np.nan
len(df)


/tmp/ipykernel_2459/3509618284.py:16: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df["A"][::10] = np.nan


1000000

In [14]:
%%time
df.to_csv('test.csv', index=False)

CPU times: user 4.58 s, sys: 102 ms, total: 4.68 s
Wall time: 4.72 s


In [15]:
%%time
df.to_hdf('test.h5', key="test", mode="w")

<timed eval>:1: PerformanceWarning: 
your performance may suffer as PyTables will pickle object types that it cannot
map directly to c-types [inferred_type->mixed,key->block0_values] [items->Index(['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J'], dtype='object')]



CPU times: user 2.77 s, sys: 663 ms, total: 3.43 s
Wall time: 3.44 s


In [16]:
%%time
df.to_feather('test.feather')

CPU times: user 1.09 s, sys: 226 ms, total: 1.32 s
Wall time: 1.13 s


In [17]:
%%time
df.to_parquet('test.parquet')

CPU times: user 2.01 s, sys: 173 ms, total: 2.18 s
Wall time: 2.26 s


In [18]:
%%time
df = pd.read_csv("test.csv")
len(df)

CPU times: user 1.45 s, sys: 105 ms, total: 1.56 s
Wall time: 1.6 s


1000000

In [19]:
%%time
df = pd.read_hdf("test.h5")
len(df)

CPU times: user 899 ms, sys: 670 ms, total: 1.57 s
Wall time: 1.57 s


1000000

In [20]:
%%time
df = pd.read_feather('test.feather')
len(df)

CPU times: user 1.41 s, sys: 379 ms, total: 1.79 s
Wall time: 1.74 s


1000000

In [21]:
%%time
df = pd.read_parquet('test.parquet')
len(df)


CPU times: user 2.04 s, sys: 448 ms, total: 2.49 s
Wall time: 2.04 s


1000000

In [22]:
df.head(10)

,A,B,C,D,E,F,G,H,I,J
0,None,0.4001572083672233,0.9787379841057392,2.240893199201458,1.8675579901499675,0,4,0,0,modi
1,-0.977277879876411,0.9500884175255894,-0.1513572082976979,-0.10321885179355784,0.41059850193837233,5,5,0,0,modi
2,0.144043571160878,1.454273506962975,0.7610377251469934,0.12167501649282841,0.44386323274542566,6,1,0,0,consectetur
3,0.33367432737426683,1.4940790731576061,-0.20515826376580087,0.31306770165090136,-0.8540957393017248,0,5,0,0,magnam
4,-2.5529898158340787,0.6536185954403606,0.8644361988595057,-0.7421650204064419,2.2697546239876076,6,7,0,0,sit
5,-1.4543656745987648,0.04575851730144607,-0.1871838500258336,1.5327792143584575,1.469358769900285,6,0,0,0,sit
6,0.1549474256969163,0.37816251960217356,-0.8877857476301128,-1.980796468223927,-0.3479121493261526,8,0,0,0,consectetur
7,0.15634896910398005,1.2302906807277207,1.2023798487844113,-0.3873268174079523,-0.30230275057533557,5,5,0,0,sed
8,-1.0485529650670926,-1.4200179371789752,-1.7062701906250126,1.9507753952317897,-0.5096521817516535,7,5,0,0,magnam
9,-0.4380743016111864,-1.2527953600499262,0.7774903558319101,-1.6138978475579515,-0.2127402802139687,2,0,0,0,sit


In [23]:
df['J'] = pd.Categorical(df.J)

In [24]:
%time df.to_feather('test.feather')


CPU times: user 844 ms, sys: 166 ms, total: 1.01 s
Wall time: 863 ms


In [25]:
%time df.to_parquet('test.parquet')

CPU times: user 1.26 s, sys: 135 ms, total: 1.39 s
Wall time: 1.4 s


In [26]:
%%time
df = pd.read_feather('test.feather')
len(df)

CPU times: user 1.19 s, sys: 338 ms, total: 1.52 s
Wall time: 1.52 s


1000000

In [27]:
%%time
df = pd.read_parquet('test.parquet')
len(df)

CPU times: user 2.29 s, sys: 471 ms, total: 2.76 s
Wall time: 2.58 s


1000000

## Feather or Parquet

- Parquet format is designed for long-term storage, where Arrow is more intended for short term or ephemeral storage because files volume are larger.
- Parquet is usually more expensive to write than Feather as it features more layers of encoding and compression.
- Feather is unmodified raw columnar Arrow memory. We will probably add simple compression to Feather in the future.
- Due to dictionary encoding, RLE encoding, and data page compression, Parquet files will often be much smaller than Feather files
- Parquet is a standard storage format for analytics that's supported by Spark. So if you are doing analytics, Parquet is a good option as a reference storage format for query by multiple systems

[source stackoverflow](https://stackoverflow.com/questions/48083405/what-are-the-differences-between-feather-and-parquet)

## Apache Arrow

[Arrow](https://arrow.apache.org/docs/python/) is a columnar in-memory analytics layer designed to accelerate big data.
It houses a set of canonical in-memory representations of
hierarchical data along with multiple language-bindings
for structure manipulation. Arrow offers an unified way to be able to
share the same data representation among languages and it will certainly be
the next standard to store dataframes in all languages.

- [R package](https://cran.r-project.org/web/packages/arrow/index.html)
- [Julia package](https://github.com/JuliaData/Arrow.jl)
- [GitHub project](https://github.com/apache/arrow)

![](https://github.com/pnavaro/big-data/blob/master/notebooks/images/arrow_ecosystem.png?raw=1)

Apache Arrow is an ideal in-memory transport layer for data that is being read or written with Parquet files. [PyArrow](https://arrow.apache.org/docs/python/) includes Python bindings to read and write Parquet files with pandas.

- columnar storage, only read the data of interest
- efficient binary packing
- choice of compression algorithms and encoding
- split data into files, allowing for parallel processing
- range of logical types
- statistics stored in metadata allow for skipping unneeded chunks
- data partitioning using the directory structure

![arrow](https://github.com/pnavaro/big-data/blob/master/notebooks/images/arrow.png?raw=1)

- https://arrow.apache.org/docs/python/csv.html
- https://arrow.apache.org/docs/python/feather.html
- https://arrow.apache.org/docs/python/parquet.html

Example:

```py
import pyarrow as pa
import pyarrow.parquet as pq
import pandas as pd
import numpy as np
arr = np.random.randn(500000) # 10% nulls
arr[::10] = np.nan
df = pd.DataFrame({'column_{0}'.format(i): arr for i in range(10)})

hdfs = pa.hdfs.connect()
table = pa.Table.from_pandas(df)
pq.write_to_dataset(table, root_path="test", filesystem=hdfs)
hdfs.ls("test")

```

### Read CSV from HDFS

Put the file test.csv on hdfs system

```python
from pyarrow import csv
with hdfs.open("/data/nycflights/1999.csv", "rb") as f:
 df = pd.read_csv(f, nrows = 10)
print(df.head())
```

### Read Parquet File from HDFS with pandas

```python
import pandas as pd
wikipedia = pd.read_parquet("hdfs://svmass2.mass.uhb.fr:54310/data/pagecounts-parquet/part-00007-8575060f-6b57-45ea-bf1d-cd77b6141f70.snappy.parquet", engine=’pyarrow’)
print(wikipedia.head())
```
### Read Parquet File with pyarrow

```py
table = pq.read_table("example.parquet")
```

### Writing a parquet file from Apache Arrow
```py
pq.write_table(table, "example.parquet")
```

### Check metadata
```py
parquet_file = pq.ParquetFile("example.parquet")
print(parquet_file.metadata)
```

### See schema
```py
print(parquet_file.schema)
```

### Connect to the Hadoop file system

```py
hdfs = pa.hdfs.connect()

# copy to local
with hdfs.open("user.txt", "rb") as f:
    f.download("user.text")

# write parquet file on hdfs
with open("example.parquet", "rb") as f:
    pa.HadoopFileSystem.upload(hdfs, "example.parquet", f)

# List files
for f in hdfs.ls("/user/navaro_p"):
    print(f)

# create a small dataframe and write it to hadoop file system
small_df = pd.DataFrame(np.array([[1, 2, 3], [4, 5, 6], [7, 8, 9]]), columns=['a', 'b', 'c'])
table = pa.Table.from_pandas(small_df)
pq.write_table(table, "small_df.parquet", filesystem=hdfs)


# Read files from Hadoop with pandas
with hdfs.open("/data/irmar.csv") as f:
    df = pd.read_csv(f)

print(df.head())

# Read parquet file from Hadoop with pandas
server = "hdfs://svmass2.mass.uhb.fr:54310"
path = "data/pagecounts-parquet/part-00007-8575060f-6b57-45ea-bf1d-cd77b6141f70.snappy.parquet"
pagecount = pd.read_parquet(os.path.join(server, path), engine="pyarrow")
print(pagecount.head())

# Read parquet file from Hadoop with pyarrow
table = pq.read_table(os.path.join(server,path))
print(table.schema)
df = table.to_pandas()
print(df.head())
```

### Exercise

- Take the second dataframe with string as last column
- Create an arrow table from pandas dataframe
- Write the file test.parquet from arrow table
- Print metadata from this parquet file
- Print schema
- Upload the file to hadoop file system
- Read this file from hadoop file system and print dataframe head


Hint: check the doc https://arrow.apache.org/docs/python/parquet.html

## Jawaban Cell Terakhir — Benchmark Format File Big Data

Bagian ini melanjutkan exercise pada cell terakhir dan disesuaikan dengan materi **Teknologi Penyimpanan Big Data**.

Fokus pengujian:
1. membuat **Arrow Table** dari pandas dataframe,
2. menulis dan membaca **Parquet**,
3. melihat **metadata** dan **schema**,
4. simulasi penyimpanan ke storage,
5. membandingkan format **CSV, HDF5, Feather, Parquet, AVRO, dan ORC** berdasarkan waktu read/write dan ukuran file,
6. memberi penjelasan pro/cons setiap format.

Materi menyebut format Big Data dibutuhkan untuk mempercepat read/write, mendukung splitting/chunking/blocking, kompresi, dan perubahan schema.


In [28]:
# Install library yang diperlukan di Google Colab
!pip -q install pyarrow fastavro tables


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 111.4 MB/s eta 0:00:00


In [29]:
import os
import time
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.orc as orc
from fastavro import writer, reader, parse_schema


### 1. Menyiapkan Data Benchmark

Data yang digunakan adalah dataframe kedua dari notebook, yaitu dataframe yang memiliki kolom numerik dan satu kolom string.  
Agar aman untuk RAM Colab/laptop, data dibatasi maksimal **100.000 row**.


In [30]:
# Gunakan dataframe yang sudah dibuat di notebook.
# Jika runtime di-reset dan df belum ada, buat ulang data contoh yang sejenis.
try:
    df
except NameError:
    np.random.seed(0)
    n = 100_000
    words = np.array(["big", "data", "storage", "parquet", "avro", "orc", "csv"])

    df = pd.DataFrame({
        "A": np.random.randn(n),
        "B": np.random.randn(n),
        "C": np.random.randn(n),
        "D": np.random.randn(n),
        "E": np.random.randn(n),
        "F": np.random.randint(0, 10, n),
        "G": np.random.randint(0, 10, n),
        "H": np.random.randint(0, 2, n),
        "I": np.random.randint(0, 2, n),
        "J": np.random.choice(words, n)
    })

    df.loc[::10, "A"] = np.nan

SAMPLE_N = min(100_000, len(df))
df_benchmark = df.head(SAMPLE_N).copy().reset_index(drop=True)

# Rapikan tipe data supaya aman untuk Arrow, Parquet, ORC, dan AVRO
for col in list("ABCDEFGHI"):
    if col in df_benchmark.columns:
        df_benchmark[col] = pd.to_numeric(df_benchmark[col], errors="coerce")

if "J" in df_benchmark.columns:
    df_benchmark["J"] = df_benchmark["J"].astype(str)

print("Jumlah data benchmark:", len(df_benchmark))
print(df_benchmark.dtypes)
df_benchmark.head()


Jumlah data benchmark: 100000
A    float64
B    float64
C    float64
D    float64
E    float64
F      int64
G      int64
H      int64
I      int64
J     object
dtype: object


,A,B,C,D,E,F,G,H,I,J
0,NaN,0.400157,0.978738,2.240893,1.867558,0,4,0,0,modi
1,-0.977278,0.950088,-0.151357,-0.103219,0.410599,5,5,0,0,modi
2,0.144044,1.454274,0.761038,0.121675,0.443863,6,1,0,0,consectetur
3,0.333674,1.494079,-0.205158,0.313068,-0.854096,0,5,0,0,magnam
4,-2.552990,0.653619,0.864436,-0.742165,2.269755,6,7,0,0,sit


### 2. Arrow Table dan Parquet

Apache Arrow dipakai sebagai representasi data kolumnar di memory.  
Dari Arrow Table, data ditulis ke format **Parquet** karena Parquet cocok untuk analitik Big Data, mendukung kompresi, metadata, dan schema.


In [31]:
# Membuat Arrow Table dari pandas dataframe
table = pa.Table.from_pandas(df_benchmark, preserve_index=False)

# Menulis Arrow Table ke Parquet dengan kompresi snappy
pq.write_table(table, "test_arrow.parquet", compression="snappy")

print("File Parquet berhasil dibuat:", os.path.exists("test_arrow.parquet"))
print("Ukuran file:", round(os.path.getsize("test_arrow.parquet") / (1024**2), 3), "MB")


File Parquet berhasil dibuat: True
Ukuran file: 4.871 MB


In [32]:
# Membaca kembali file Parquet
table_read = pq.read_table("test_arrow.parquet")
df_read = table_read.to_pandas()

print("Jumlah row hasil read:", len(df_read))
df_read.head()


Jumlah row hasil read: 100000


,A,B,C,D,E,F,G,H,I,J
0,NaN,0.400157,0.978738,2.240893,1.867558,0,4,0,0,modi
1,-0.977278,0.950088,-0.151357,-0.103219,0.410599,5,5,0,0,modi
2,0.144044,1.454274,0.761038,0.121675,0.443863,6,1,0,0,consectetur
3,0.333674,1.494079,-0.205158,0.313068,-0.854096,0,5,0,0,magnam
4,-2.552990,0.653619,0.864436,-0.742165,2.269755,6,7,0,0,sit


In [33]:
# Melihat metadata dan schema Parquet
parquet_file = pq.ParquetFile("test_arrow.parquet")

print("=== PARQUET METADATA ===")
print(parquet_file.metadata)

print("\n=== PARQUET SCHEMA ===")
print(parquet_file.schema)


=== PARQUET METADATA ===
  created_by: parquet-cpp-arrow version 18.1.0
  num_columns: 10
  num_rows: 100000
  num_row_groups: 1
  format_version: 2.6
  serialized_size: 4688

=== PARQUET SCHEMA ===
required group field_id=-1 schema {
  optional double field_id=-1 A;
  optional double field_id=-1 B;
  optional double field_id=-1 C;
  optional double field_id=-1 D;
  optional double field_id=-1 E;
  optional int64 field_id=-1 F;
  optional int64 field_id=-1 G;
  optional int64 field_id=-1 H;
  optional int64 field_id=-1 I;
  optional binary field_id=-1 J (String);
}



### 3. Simulasi Storage

Pada materi, penyimpanan Big Data dapat berupa filesystem lokal, filesystem terdistribusi seperti HDFS, atau cloud storage seperti S3/Azure Blob/Google Drive.  
Karena Colab biasa tidak memiliki cluster HDFS, bagian ini dibuat sebagai **simulasi storage lokal**.


In [34]:
# Simulasi upload/copy ke folder storage
storage_dir = "storage_simulation"
os.makedirs(storage_dir, exist_ok=True)

src_file = "test_arrow.parquet"
dst_file = os.path.join(storage_dir, "test_arrow.parquet")

shutil.copy(src_file, dst_file)

print("File berhasil disimpan ke:", dst_file)
print("Daftar file storage:", os.listdir(storage_dir))

# Membaca file dari storage
df_from_storage = pd.read_parquet(dst_file)
df_from_storage.head()


File berhasil disimpan ke: storage_simulation/test_arrow.parquet
Daftar file storage: ['test_arrow.parquet']


,A,B,C,D,E,F,G,H,I,J
0,NaN,0.400157,0.978738,2.240893,1.867558,0,4,0,0,modi
1,-0.977278,0.950088,-0.151357,-0.103219,0.410599,5,5,0,0,modi
2,0.144044,1.454274,0.761038,0.121675,0.443863,6,1,0,0,consectetur
3,0.333674,1.494079,-0.205158,0.313068,-0.854096,0,5,0,0,magnam
4,-2.552990,0.653619,0.864436,-0.742165,2.269755,6,7,0,0,sit


### 4. Benchmark Format File Big Data

Format yang dibandingkan:
- **CSV** sebagai format teks umum,
- **HDF5** sebagai format penyimpanan hierarkis,
- **Feather** sebagai format Arrow untuk pertukaran dataframe cepat,
- **Parquet** sebagai format kolumnar Big Data,
- **AVRO** sebagai format biner dengan schema,
- **ORC** sebagai format kolumnar yang banyak digunakan di ekosistem Hadoop/Hive.

Metrik yang dihitung:
1. waktu tulis file,
2. waktu baca file,
3. ukuran file.


In [35]:
def file_size_mb(path):
    return os.path.getsize(path) / (1024**2)

def measure_time(func):
    start = time.perf_counter()
    result = func()
    elapsed = time.perf_counter() - start
    return elapsed, result

def clean_file(path):
    if os.path.exists(path):
        os.remove(path)

def avro_schema_from_dataframe(dataframe):
    fields = []

    for col in dataframe.columns:
        if pd.api.types.is_integer_dtype(dataframe[col]):
            avro_type = ["null", "long"]
        elif pd.api.types.is_float_dtype(dataframe[col]):
            avro_type = ["null", "double"]
        else:
            avro_type = ["null", "string"]

        fields.append({
            "name": str(col),
            "type": avro_type,
            "default": None
        })

    return {
        "type": "record",
        "name": "BenchmarkRecord",
        "namespace": "bigdata.storage",
        "fields": fields
    }

def dataframe_to_avro_records(dataframe):
    # Ubah NaN menjadi None agar sesuai union null di AVRO
    safe_df = dataframe.astype(object).where(pd.notnull(dataframe), None)
    return safe_df.to_dict(orient="records")

results = []


#### Benchmark CSV

In [ ]:
# CSV
try:
    path = "benchmark.csv"
    clean_file(path)

    write_time, _ = measure_time(lambda: df_benchmark.to_csv(path, index=False))
    read_time, _ = measure_time(lambda: pd.read_csv(path))

    results.append({
        "format": "CSV",
        "write_time_sec": write_time,
        "read_time_sec": read_time,
        "size_mb": file_size_mb(path),
        "status": "OK"
    })
except Exception as e:
    results.append({
        "format": "CSV",
        "write_time_sec": None,
        "read_time_sec": None,
        "size_mb": None,
        "status": str(e)
    })


#### Benchmark HDF5

In [ ]:
# HDF5
try:
    path = "benchmark.h5"
    clean_file(path)

    write_time, _ = measure_time(lambda: df_benchmark.to_hdf(path, key="data", mode="w"))
    read_time, _ = measure_time(lambda: pd.read_hdf(path, key="data"))

    results.append({
        "format": "HDF5",
        "write_time_sec": write_time,
        "read_time_sec": read_time,
        "size_mb": file_size_mb(path),
        "status": "OK"
    })
except Exception as e:
    results.append({
        "format": "HDF5",
        "write_time_sec": None,
        "read_time_sec": None,
        "size_mb": None,
        "status": str(e)
    })


#### Benchmark Feather

In [ ]:
# Feather
try:
    path = "benchmark.feather"
    clean_file(path)

    write_time, _ = measure_time(lambda: df_benchmark.to_feather(path))
    read_time, _ = measure_time(lambda: pd.read_feather(path))

    results.append({
        "format": "Feather",
        "write_time_sec": write_time,
        "read_time_sec": read_time,
        "size_mb": file_size_mb(path),
        "status": "OK"
    })
except Exception as e:
    results.append({
        "format": "Feather",
        "write_time_sec": None,
        "read_time_sec": None,
        "size_mb": None,
        "status": str(e)
    })


#### Benchmark Parquet

In [ ]:
# Parquet dengan PyArrow
try:
    path = "benchmark.parquet"
    clean_file(path)

    def write_parquet():
        table = pa.Table.from_pandas(df_benchmark, preserve_index=False)
        pq.write_table(table, path, compression="snappy")

    def read_parquet():
        return pq.read_table(path).to_pandas()

    write_time, _ = measure_time(write_parquet)
    read_time, _ = measure_time(read_parquet)

    results.append({
        "format": "Parquet",
        "write_time_sec": write_time,
        "read_time_sec": read_time,
        "size_mb": file_size_mb(path),
        "status": "OK"
    })
except Exception as e:
    results.append({
        "format": "Parquet",
        "write_time_sec": None,
        "read_time_sec": None,
        "size_mb": None,
        "status": str(e)
    })


#### Benchmark AVRO

In [ ]:
# AVRO dengan fastavro
try:
    path = "benchmark.avro"
    clean_file(path)

    schema = parse_schema(avro_schema_from_dataframe(df_benchmark))
    records = dataframe_to_avro_records(df_benchmark)

    def write_avro():
        with open(path, "wb") as out:
            writer(out, schema, records)

    def read_avro():
        with open(path, "rb") as fo:
            return pd.DataFrame(list(reader(fo)))

    write_time, _ = measure_time(write_avro)
    read_time, _ = measure_time(read_avro)

    results.append({
        "format": "AVRO",
        "write_time_sec": write_time,
        "read_time_sec": read_time,
        "size_mb": file_size_mb(path),
        "status": "OK"
    })
except Exception as e:
    results.append({
        "format": "AVRO",
        "write_time_sec": None,
        "read_time_sec": None,
        "size_mb": None,
        "status": str(e)
    })


#### Benchmark ORC

In [ ]:
# ORC dengan PyArrow
try:
    path = "benchmark.orc"
    clean_file(path)

    def write_orc():
        table = pa.Table.from_pandas(df_benchmark, preserve_index=False)
        orc.write_table(table, path, compression="snappy")

    def read_orc():
        return orc.read_table(path).to_pandas()

    write_time, _ = measure_time(write_orc)
    read_time, _ = measure_time(read_orc)

    results.append({
        "format": "ORC",
        "write_time_sec": write_time,
        "read_time_sec": read_time,
        "size_mb": file_size_mb(path),
        "status": "OK"
    })
except Exception as e:
    results.append({
        "format": "ORC",
        "write_time_sec": None,
        "read_time_sec": None,
        "size_mb": None,
        "status": str(e)
    })


#### Tabel Hasil Benchmark

In [ ]:
# Tabel hasil benchmark semua format
result_df = pd.DataFrame(results)
result_df


In [ ]:
# Ambil hasil yang berhasil dijalankan
ok_df = result_df[result_df["status"] == "OK"].copy()

ok_df[["write_time_sec", "read_time_sec", "size_mb"]] = ok_df[
    ["write_time_sec", "read_time_sec", "size_mb"]
].round(4)

ok_df


In [ ]:
# Grafik perbandingan waktu read/write
plt.figure(figsize=(10, 5))

x = np.arange(len(ok_df))
width = 0.35

plt.bar(x - width/2, ok_df["write_time_sec"], width, label="Write Time")
plt.bar(x + width/2, ok_df["read_time_sec"], width, label="Read Time")

plt.xticks(x, ok_df["format"])
plt.xlabel("Format File")
plt.ylabel("Waktu (detik)")
plt.title("Perbandingan Kinerja Waktu Read/Write Format File Big Data")
plt.legend()
plt.grid(axis="y", alpha=0.3)
plt.show()


In [ ]:
# Grafik perbandingan ukuran file
plt.figure(figsize=(10, 5))

plt.bar(ok_df["format"], ok_df["size_mb"])

plt.xlabel("Format File")
plt.ylabel("Ukuran File (MB)")
plt.title("Perbandingan Ukuran File Setiap Format")
plt.grid(axis="y", alpha=0.3)
plt.show()


### 5. Pro dan Cons Setiap Format

| Format | Pro | Cons |
|---|---|---|
| CSV | Mudah dibaca manusia, universal, bisa dibuka banyak aplikasi | Ukuran besar, tidak menyimpan schema, read/write lambat untuk data besar |
| HDF5 | Cocok untuk data numerik besar dan penyimpanan hierarkis | Tidak seumum Parquet di ekosistem Hadoop/Spark |
| Feather | Sangat cepat untuk read/write dataframe karena berbasis Arrow | Kurang cocok untuk penyimpanan jangka panjang dan kompresi tidak sekuat Parquet |
| Parquet | Kolumnar, mendukung schema, kompresi, metadata, dan efisien untuk analitik Big Data | Write bisa lebih mahal karena encoding dan kompresi |
| AVRO | Menyimpan schema/metadata, cocok untuk data streaming dan perubahan schema | Format biner, tidak mudah dibaca langsung oleh manusia |
| ORC | Kolumnar, kompresi tinggi, efisien untuk query analitik di Hadoop/Hive | Dukungan umum di luar ekosistem Hadoop tidak seluas Parquet |
